In [1]:
!pip install -q wandb accelerate

In [2]:
import os
base_path = '/kaggle/working'
project_name = 'weight-vs-loss-convergence-pinn'
project_root = os.path.join(base_path, project_name)

if not os.path.exists(project_root):
    os.chdir(base_path)
    !git clone https://github.com/nthday-jpg/weight-vs-loss-convergence-pinn.git
    print("Repository cloned successfully!")
else:
    os.chdir(project_root)
    !git pull
    print("Repository updated successfully!")
%cd {base_path}

# Ensure project root is in PYTHONPATH for script's imports
os.environ['PYTHONPATH'] = project_root

Cloning into 'weight-vs-loss-convergence-pinn'...
remote: Enumerating objects: 129, done.
remote: Counting objects: 100% (129/129), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 129 (delta 73), reused 94 (delta 40), pack-reused 0 (from 0)
Receiving objects: 100% (129/129), 143.10 KiB | 5.30 MiB/s, done.
Resolving deltas: 100% (73/73), done.
Repository cloned successfully!
/kaggle/working


In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
WANDB_KEY = user_secrets.get_secret("WANDB_KEY")

!wandb login {WANDB_KEY}

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [4]:
# Generate data
save_dir = os.path.join(project_root, 'data')
!python -m burgers.data.gen_data \
        --save_dir {save_dir} \
        --nn 511\
        --steps 200 \
        --nu 0.01\
        --t_final 1.0

Solving Burgers equation with 511 spatial points and 200 time steps...
Viscosity nu = 0.010000
Integration successful: True
Number of function evaluations: 13694
Data saved to /kaggle/working/weight-vs-loss-convergence-pinn/data/burgers.pt
Data saved to /kaggle/working/weight-vs-loss-convergence-pinn/data/burgers.npz

Done! Data shape: (201, 511)
Time range: [0.000, 1.000]
Space range: [-1.000, 1.000]
Solution range: [-0.999981, 0.999981]

Total time: 2.36 seconds


In [5]:
import json

config = {
    'layers': [2, 64, 1],
    'learning_rate': 1e-2,
    'num_epochs': 500,
    'step_per_epoch': 100,
    'batch_size': 32768,
    'l2_reg': 1e-6,
    'log_interval': 1,
    'balancer_type': "uniform",
    'wandb_project': "loss-weight-convergence",
    'wandb_run_name': 'unifom'
    
}

# Save config to JSON file
config_path = os.path.join(project_root, 'config.json')
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)


In [6]:

! accelerate launch weight-vs-loss-convergence-pinn/burgers/train.py \
    --config_file /kaggle/working/weight-vs-loss-convergence-pinn/config.json \
    --data_file /kaggle/working/weight-vs-loss-convergence-pinn/data/burgers.pt

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `2`
		More than one GPU was found, enabling multi-GPU training.
		If this was unintended please pass in `--num_processes=1`.
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: nthday-jpg (nthday-jpg-hcmus-vnu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ setting up run 4jp9y4zd (0.2s)
wandb: Tracking run with wandb version 0.24.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260308_150225-4jp9y4zd
wandb: Run `wandb offline` t